In [3]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'gdown'])

from google.oauth2 import service_account
import googleapiclient

print("Testing internet...")
import urllib.request
urllib.request.urlopen('http://google.com')
print("✅ Internet works")

import os
print(f"✅ GPU available: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")

Testing internet...
✅ Internet works
✅ GPU available: Tesla T4
Tesla T4


In [4]:
import os

# Install PyDrive2 for Drive access
os.system('pip install -q PyDrive2')

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
print("✅ Drive authenticated")

✅ Drive authenticated


In [8]:
import subprocess, os

# Find the file IDs for our zips
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# Search for our zip files
files = drive.ListFile({
    'q': "title contains 'TrainVal' and mimeType='application/zip' and trashed=false"
}).GetList()

for f in files:
    print(f"Name: {f['title']}")
    print(f"ID:   {f['id']}")
    print()

In [9]:
# List everything in your dlp_project folder to find the folder IDs
file_list = drive.ListFile({
    'q': "title='dlp_project' and trashed=false"
}).GetList()

for f in file_list:
    print(f"Name: {f['title']}")
    print(f"ID:   {f['id']}")
    print(f"Type: {f['mimeType']}")
    print()

Name: dlp_project
ID:   1hNdBtf8VQrnLQuBro4U4IR1pD4eA3MQ3
Type: application/vnd.google-apps.folder



In [10]:
def list_folder(folder_id, indent=0):
    files = drive.ListFile({
        'q': f"'{folder_id}' in parents and trashed=false"
    }).GetList()
    for f in files:
        print(" " * indent + f"[{f['mimeType'].split('.')[-1]}] {f['title']} — {f['id']}")
        if 'folder' in f['mimeType']:
            list_folder(f['id'], indent + 4)

list_folder('1hNdBtf8VQrnLQuBro4U4IR1pD4eA3MQ3')

[shortcut] DLP Project Datasets — 19pAUFicYK9AtpRiT1R2D171mogUSBo0y
[folder] checkpoints — 1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8


In [11]:
# Resolve the shortcut to get the real folder ID
shortcut = drive.CreateFile({'id': '19pAUFicYK9AtpRiT1R2D171mogUSBo0y'})
shortcut.FetchMetadata(fields='shortcutDetails')
real_id = shortcut['shortcutDetails']['targetId']
print(f"Real folder ID: {real_id}")

# Now list contents of the real folder
list_folder(real_id)

Real folder ID: 1KcI6TFgaSQ1BXQzF4HqzSAQT2n9ygTxd
[folder] checkpoints — 18cprgjVtQcEm4B9g-S9N0wpjHziGlUS5
    [application/x-zip] sd_inpaint_resume.pth — 1HDX0fbYVgwuasn70VLNH7x0dFcwOs4rj
[folder] DeepFashion — 1GcUoyoTdMFl3uC1Bolk1WYcHF3k98zxP
    [folder] Img — 1rHQ1IT8ogjsz_C0WQUILjB3RpvaFmNPQ
        [application/x-zip-compressed] img.zip — 1VDmSzYS4jeH-45c7JMVF22bTrqzdW0Gp
    [application/x-zip-compressed] img_highres_seg.zip — 1wNZY1blOmOThcCuCRyD36HsncKp4z5wM
    [text/markdown] README_DeepFashion_segmentation.md — 1doDZ9cRmJ4EGhSPLiiUaTxtrTAFd94p_
    [text/plain] README_DeepFashion_main.txt — 16LyhJx6sEWAA2K3IBzueTDR_Zkw_25Rx
    [text/plain] list_eval_partition.txt — 115QWWX_yttJYICGuvHw7wAf6y7Ahj1P3
    [text/plain] list_item_inshop.txt — 1zRgrGhw7C2NWPPtfXA14hyLngTd7kcNY
[folder] Look Into Person — 13ukRItUW1BGPeT8CIrEhGoQatDisuWjn
    [folder] TrainVal_images — 1omgM-_7wgO8kAnknbBgLrt7SAsDGxOZF
        [text/plain] val_id.txt — 1BSCGeJX6KNICWvSj17EzEtMQiCGrP21e
        [

In [12]:
import os

os.makedirs('/content/lip/images', exist_ok=True)
os.makedirs('/content/lip/annotations', exist_ok=True)
os.makedirs('/content/deepfashion', exist_ok=True)

files_to_download = {
    'TrainVal_images.zip':              ('11056WM2HM1C2WGzOWoA1NnIwPucbM0Qa', '/content/lip/images/'),
    'TrainVal_parsing_annotations.zip': ('1srpJV9OIT7C-qK-FI4ifETzSVdSGviRk', '/content/lip/annotations/'),
    'train_id.txt':                     ('1-0lHlpNwGqH-CbOBml07eMv5zc1NCMXg', '/content/lip/'),
    'val_id.txt':                       ('1BSCGeJX6KNICWvSj17EzEtMQiCGrP21e',  '/content/lip/'),
}

for name, (file_id, dest) in files_to_download.items():
    out_path = dest + name
    if os.path.exists(out_path):
        print(f"✅ Already exists: {name}")
        continue
    print(f"Downloading {name}...")
    f = drive.CreateFile({'id': file_id})
    f.GetContentFile(out_path)
    size = os.path.getsize(out_path) // (1024*1024)
    print(f"✅ Done — {size}MB → {out_path}")

✅ Done — 628MB → /content/lip/images/TrainVal_images.zip
✅ Done — 87MB → /content/lip/annotations/TrainVal_parsing_annotations.zip
✅ Done — 0MB → /content/lip/train_id.txt
✅ Done — 0MB → /content/lip/val_id.txt


In [13]:
import zipfile

extractions = [
    ('/content/lip/images/TrainVal_images.zip', '/content/lip/images/'),
    ('/content/lip/annotations/TrainVal_parsing_annotations.zip', '/content/lip/annotations/'),
]

for zip_path, extract_to in extractions:
    print(f"Extracting {os.path.basename(zip_path)}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    print(f"✅ Done → {extract_to}")

# Verify
dirs_to_check = [
    '/content/lip/images/train_images',
    '/content/lip/images/val_images',
    '/content/lip/annotations/train_segmentations',
    '/content/lip/annotations/val_segmentations',
]

print("\nVerifying:")
for d in dirs_to_check:
    count = len(os.listdir(d)) if os.path.exists(d) else 0
    print(f"{'✅' if count > 0 else '❌'} {d} — {count} files")


Extracting TrainVal_images.zip...
✅ Done → /content/lip/images/
Extracting TrainVal_parsing_annotations.zip...
✅ Done → /content/lip/annotations/

Verifying:
✅ /content/lip/images/train_images — 30462 files
✅ /content/lip/images/val_images — 10000 files
✅ /content/lip/annotations/train_segmentations — 30462 files
✅ /content/lip/annotations/val_segmentations — 10000 files


In [14]:
!git clone https://github.com/nabirakhan/luxe /content/Luxe

Cloning into '/content/Luxe'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 141 (delta 47), reused 100 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 10.68 MiB | 32.65 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [15]:
import sys
sys.path.insert(0, '/content/Luxe/backend')

from data.lip_loader import get_lip_loaders

train_loader, val_loader = get_lip_loaders(
    '/content/lip/images',
    '/content/lip/annotations',
    batch_size=2
)

print("\n--- Train batch ---")
imgs, labels = next(iter(train_loader))
print(f"imgs shape:   {imgs.shape}")
print(f"labels shape: {labels.shape}")
print(f"label values: {labels.unique().tolist()}")

print("\n--- Val batch ---")
imgs, labels = next(iter(val_loader))
print(f"imgs shape:   {imgs.shape}")
print(f"labels shape: {labels.shape}")
print(f"label values: {labels.unique().tolist()}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dlp_project/DLP Project Datasets/Look Into Person/TrainVal_images/train_id.txt'

In [17]:
fixed_loader = '''"""Look Into Person (LIP) dataset loader for SegFormer training.

Zip extracts to:
  /content/lip/images/train_images/*.jpg          (30,462 images)
  /content/lip/images/val_images/*.jpg            (10,000 images)
  /content/lip/annotations/train_segmentations/*.png  (30,462 masks)
  /content/lip/annotations/val_segmentations/*.png    (10,000 masks)

Train/val split from train_id.txt and val_id.txt.
"""

import numpy as np
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

HIGH_ATTACK_CLASSES = {5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17}

# Works on both Colab and Kaggle
def _find_id_txt(filename):
    candidates = [
        f"/content/lip/{filename}",
        f"/content/drive/MyDrive/dlp_project/DLP Project Datasets/Look Into Person/TrainVal_images/{filename}",
    ]
    for p in candidates:
        if Path(p).exists():
            return p
    raise FileNotFoundError(f"{filename} not found. Tried: {candidates}")


def _load_stems(txt_path):
    with open(txt_path, "r") as f:
        return [line.strip() for line in f if line.strip()]


class LIPDataset(Dataset):
    def __init__(self, img_dir: str, ann_dir: str, split: str = "train"):
        if split == "train":
            self.img_dir = Path(img_dir) / "train_images"
            self.ann_dir = Path(ann_dir) / "train_segmentations"
            id_txt = _find_id_txt("train_id.txt")
        else:
            self.img_dir = Path(img_dir) / "val_images"
            self.ann_dir = Path(ann_dir) / "val_segmentations"
            id_txt = _find_id_txt("val_id.txt")

        all_stems = _load_stems(id_txt)
        available = {p.stem for p in self.img_dir.glob("*.jpg")}
        self.stems = [s for s in all_stems if s in available]
        print(f"[LIPDataset] split={split}  found={len(self.stems)}/{len(all_stems)}")

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]

        img = Image.open(self.img_dir / f"{stem}.jpg").convert("RGB").resize((512, 512), Image.LANCZOS)
        img_tensor = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).permute(2, 0, 1)

        ann_path = self.ann_dir / f"{stem}.png"
        if ann_path.exists():
            label_np = np.array(Image.open(ann_path).resize((512, 512), Image.NEAREST), dtype=np.int64)
            label_tensor = torch.from_numpy(label_np).unsqueeze(0)
        else:
            label_tensor = torch.zeros(1, 512, 512, dtype=torch.long)

        return img_tensor, label_tensor


class LIPBinaryDataset(LIPDataset):
    def __getitem__(self, idx):
        img_tensor, label_tensor = super().__getitem__(idx)
        binary = torch.zeros_like(label_tensor, dtype=torch.float32)
        for cls in HIGH_ATTACK_CLASSES:
            binary[label_tensor == cls] = 1.0
        return img_tensor, binary


def get_lip_loaders(img_dir, ann_dir, batch_size=8, binary=False):
    cls = LIPBinaryDataset if binary else LIPDataset
    train_ds = cls(img_dir, ann_dir, split="train")
    val_ds   = cls(img_dir, ann_dir, split="val")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader
'''

with open('/content/Luxe/backend/data/lip_loader.py', 'w') as f:
    f.write(fixed_loader)

print("✅ lip_loader.py updated for Kaggle compatibility")

✅ lip_loader.py updated for Kaggle compatibility


In [18]:
import importlib
import data.lip_loader as lip_module
importlib.reload(lip_module)
from data.lip_loader import get_lip_loaders

train_loader, val_loader = get_lip_loaders(
    '/content/lip/images',
    '/content/lip/annotations',
    batch_size=2
)

print("\n--- Train batch ---")
imgs, labels = next(iter(train_loader))
print(f"imgs shape:   {imgs.shape}")
print(f"labels shape: {labels.shape}")
print(f"label values: {labels.unique().tolist()}")

print("\n--- Val batch ---")
imgs, labels = next(iter(val_loader))
print(f"imgs shape:   {imgs.shape}")
print(f"labels shape: {labels.shape}")
print(f"label values: {labels.unique().tolist()}")

[LIPDataset] split=train  found=30462/30462
[LIPDataset] split=val  found=10000/10000

--- Train batch ---
imgs shape:   torch.Size([2, 3, 512, 512])
labels shape: torch.Size([2, 1, 512, 512])
label values: [0, 1, 5, 7, 13, 14]

--- Val batch ---
imgs shape:   torch.Size([2, 3, 512, 512])
labels shape: torch.Size([2, 1, 512, 512])
label values: [0, 1, 5, 6, 7, 8, 13, 14, 16, 17, 18, 19]


In [19]:
with open('/content/Luxe/backend/data/lip_loader.py', 'r') as f:
    print(f.read())

"""Look Into Person (LIP) dataset loader for SegFormer training.

Zip extracts to:
  /content/lip/images/train_images/*.jpg          (30,462 images)
  /content/lip/images/val_images/*.jpg            (10,000 images)
  /content/lip/annotations/train_segmentations/*.png  (30,462 masks)
  /content/lip/annotations/val_segmentations/*.png    (10,000 masks)

Train/val split from train_id.txt and val_id.txt.
"""

import numpy as np
import torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

HIGH_ATTACK_CLASSES = {5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17}

# Works on both Colab and Kaggle
def _find_id_txt(filename):
    candidates = [
        f"/content/lip/{filename}",
        f"/content/drive/MyDrive/dlp_project/DLP Project Datasets/Look Into Person/TrainVal_images/{filename}",
    ]
    for p in candidates:
        if Path(p).exists():
            return p
    raise FileNotFoundError(f"{filename} not found. Tried: {candidates}")


def _load_

In [21]:
!pip install -q transformers accelerate

In [22]:
import transformers, accelerate
print(f"✅ transformers: {transformers.__version__}")
print(f"✅ accelerate:   {accelerate.__version__}")

✅ transformers: 5.0.0
✅ accelerate:   1.12.0


In [23]:
import sys, os
sys.path.insert(0, '/content/Luxe/backend')

# Override the find_repo_base and find_drive_base in train_segformer
# since we're on Kaggle not Colab
os.environ['REPO_BASE'] = '/content/Luxe/backend'

!cd /content/Luxe/backend && python training/train_segformer.py

Traceback (most recent call last):
  File "/content/Luxe/backend/training/train_segformer.py", line 29, in <module>
    drive.mount('/content/drive')
  File "/usr/local/lib/python3.12/dist-packages/google/colab/drive.py", line 97, in mount
    return _mount(
           ^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/drive.py", line 115, in _mount
    raise NotImplementedError(
NotImplementedError: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.


In [24]:
import os, sys, torch
import numpy as np
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from transformers import SegformerForSemanticSegmentation
from pathlib import Path
from tqdm.auto import tqdm

sys.path.insert(0, '/content/Luxe/backend')
from data.lip_loader import get_lip_loaders

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ── Config ───────────────────────────────────────────────────────────────────
DEVICE           = torch.device('cuda')
NUM_CLASSES      = 20
EPOCHS           = 20
LR               = 6e-5
BATCH_SIZE       = 4
IMG_SIZE         = 512
DRIVE_CKPT_FOLDER_ID = '1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8'
CKPT_DIR         = Path('/content/Luxe/backend/checkpoints')
CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH        = CKPT_DIR / 'segformer_lip.pth'
RESUME_PATH      = CKPT_DIR / 'segformer_resume.pth'

# ── Drive upload helper — overwrites existing file instead of creating new ───
def upload_to_drive(local_path, filename, folder_id):
    file_list = drive.ListFile({
        'q': f"title='{filename}' and '{folder_id}' in parents and trashed=false"
    }).GetList()
    if file_list:
        f = drive.CreateFile({'id': file_list[0]['id']})
    else:
        f = drive.CreateFile({'title': filename, 'parents': [{'id': folder_id}]})
    f.SetContentFile(str(local_path))
    f.Upload()

# ── Data ─────────────────────────────────────────────────────────────────────
train_loader, val_loader = get_lip_loaders(
    '/content/lip/images',
    '/content/lip/annotations',
    batch_size=BATCH_SIZE
)
print(f"✅ Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

# ── Model ────────────────────────────────────────────────────────────────────
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/segformer-b2-finetuned-ade-512-512',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()

# ── Resume ───────────────────────────────────────────────────────────────────
start_epoch = 0
best_miou   = 0.0

if RESUME_PATH.exists():
    print(f"🔄 Resuming from {RESUME_PATH}")
    ckpt = torch.load(RESUME_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    best_miou   = ckpt['best_miou']
    print(f"✅ Resumed from epoch {start_epoch} | Best mIoU: {best_miou:.4f}")
else:
    print("🆕 Starting fresh")

# ── mIoU ─────────────────────────────────────────────────────────────────────
def compute_miou(model, val_dl, device, num_classes=NUM_CLASSES):
    intersection = np.zeros(num_classes)
    union        = np.zeros(num_classes)
    model.eval()
    with torch.no_grad():
        for imgs, anns in tqdm(val_dl, desc='  Val mIoU', leave=False):
            imgs = imgs.to(device)
            out  = model(pixel_values=imgs)
            upsampled = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE),
                                      mode='bilinear', align_corners=False)
            preds = upsampled.argmax(dim=1).cpu().numpy()
            anns  = anns.squeeze(1).numpy()
            for c in range(num_classes):
                pred_c  = preds == c
                label_c = anns  == c
                intersection[c] += (pred_c & label_c).sum()
                union[c]        += (pred_c | label_c).sum()
    iou_per_class = np.where(union > 0, intersection / union, np.nan)
    return float(np.nanmean(iou_per_class))

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, (imgs, anns) in enumerate(pbar):
        imgs = imgs.to(DEVICE)
        anns = anns.squeeze(1).to(DEVICE)

        with autocast():
            out       = model(pixel_values=imgs)
            logits_up = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE),
                                      mode='bilinear', align_corners=False)
            loss      = F.cross_entropy(logits_up, anns)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        # Mid-epoch save every 500 steps
        if (step + 1) % 500 == 0:
            torch.save({
                'epoch':     epoch,
                'model':     model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler':    scaler.state_dict(),
                'best_miou': best_miou,
            }, RESUME_PATH)
            print(f"  💾 Mid-epoch checkpoint saved (step {step+1})")

    scheduler.step()
    miou     = compute_miou(model, val_loader, DEVICE)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val mIoU: {miou:.4f}")

    # Save resume checkpoint locally
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler':    scaler.state_dict(),
        'best_miou': best_miou,
    }, RESUME_PATH)

    # Save best model locally + to Drive
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ✅ Best model saved (mIoU: {best_miou:.4f})")
        upload_to_drive(BEST_PATH, 'segformer_lip.pth', DRIVE_CKPT_FOLDER_ID)
        print(f"  ☁️  Best model uploaded to Drive")

    # Always upload resume checkpoint to Drive
    upload_to_drive(RESUME_PATH, 'segformer_resume.pth', DRIVE_CKPT_FOLDER_ID)
    print(f"  ☁️  Resume checkpoint uploaded to Drive")

print(f"\n🎉 Training complete. Best mIoU: {best_miou:.4f}")

✅ GPU: Tesla T4
[LIPDataset] split=train  found=30462/30462
[LIPDataset] split=val  found=10000/10000
✅ Train: 30462 | Val: 10000


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                    
------------------------------+----------+----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([20, 768, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([20])                      

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


...

/tmp/ipykernel_55/2062711234.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  💾 Mid-epoch checkpoint saved (step 500)
  💾 Mid-epoch checkpoint saved (step 1000)
  💾 Mid-epoch checkpoint saved (step 1500)
  💾 Mid-epoch checkpoint saved (step 2000)
  💾 Mid-epoch checkpoint saved (step 2500)
  💾 Mid-epoch checkpoint saved (step 3000)
  💾 Mid-epoch checkpoint saved (step 3500)
  💾 Mid-epoch checkpoint saved (step 4000)
  💾 Mid-epoch checkpoint saved (step 4500)
  💾 Mid-epoch checkpoint saved (step 5000)
  💾 Mid-epoch checkpoint saved (step 5500)
  💾 Mid-epoch checkpoint saved (step 6000)
  💾 Mid-epoch checkpoint saved (step 6500)
  💾 Mid-epoch checkpoint saved (step 7000)
  💾 Mid-epoch checkpoint saved (step 7500)


  Val mIoU:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 1/20 | Loss: 0.6951 | Val mIoU: 0.3907
  ✅ Best model saved (mIoU: 0.3907)


InvalidConfigError: Invalid client secrets file ('Error opening file', 'client_secrets.json', 'No such file or directory', 2)

In [25]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pathlib import Path

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

DRIVE_CKPT_FOLDER_ID = '1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8'
CKPT_DIR = Path('/content/Luxe/backend/checkpoints')

def upload_to_drive(local_path, filename, folder_id):
    file_list = drive.ListFile({
        'q': f"title='{filename}' and '{folder_id}' in parents and trashed=false"
    }).GetList()
    if file_list:
        f = drive.CreateFile({'id': file_list[0]['id']})
    else:
        f = drive.CreateFile({'title': filename, 'parents': [{'id': folder_id}]})
    f.SetContentFile(str(local_path))
    f.Upload()
    print(f"✅ Uploaded {filename}")

for fname in ['segformer_lip.pth', 'segformer_resume.pth']:
    path = CKPT_DIR / fname
    if path.exists():
        upload_to_drive(path, fname, DRIVE_CKPT_FOLDER_ID)
    else:
        print(f"❌ Not found locally: {fname}")

✅ Uploaded segformer_lip.pth
✅ Uploaded segformer_resume.pth


In [2]:
import os, sys, torch
import numpy as np
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from transformers import SegformerForSemanticSegmentation
from pathlib import Path
from tqdm.auto import tqdm

sys.path.insert(0, '/content/Luxe/backend')
from data.lip_loader import get_lip_loaders

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ── Config ───────────────────────────────────────────────────────────────────
DEVICE               = torch.device('cuda')
NUM_CLASSES          = 20
EPOCHS               = 20
LR                   = 6e-5
BATCH_SIZE           = 4
IMG_SIZE             = 512
DRIVE_CKPT_FOLDER_ID = '1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8'
CKPT_DIR             = Path('/content/Luxe/backend/checkpoints')
CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH            = CKPT_DIR / 'segformer_lip.pth'
RESUME_PATH          = CKPT_DIR / 'segformer_resume.pth'

# ── Drive upload — never crashes training ─────────────────────────────────────
def upload_to_drive(local_path, filename, folder_id):
    try:
        # Re-auth every time to avoid token expiry
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        fresh_drive = GoogleDrive(gauth)
        
        file_list = fresh_drive.ListFile({
            'q': f"title='{filename}' and '{folder_id}' in parents and trashed=false"
        }).GetList()
        if file_list:
            f = fresh_drive.CreateFile({'id': file_list[0]['id']})
        else:
            f = fresh_drive.CreateFile({'title': filename, 'parents': [{'id': folder_id}]})
        f.SetContentFile(str(local_path))
        f.Upload()
        print(f"  ☁️  Uploaded {filename} to Drive")
    except Exception as e:
        print(f"  ⚠️  Drive upload failed (training continues): {e}")
        
# ── Data ─────────────────────────────────────────────────────────────────────
train_loader, val_loader = get_lip_loaders(
    '/content/lip/images',
    '/content/lip/annotations',
    batch_size=BATCH_SIZE
)
print(f"✅ Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

# ── Model ────────────────────────────────────────────────────────────────────
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/segformer-b2-finetuned-ade-512-512',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler('cuda')

# ── Resume ───────────────────────────────────────────────────────────────────
start_epoch = 0
best_miou   = 0.0

if RESUME_PATH.exists():
    print(f"🔄 Resuming from {RESUME_PATH}")
    ckpt = torch.load(RESUME_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    best_miou   = ckpt['best_miou']
    print(f"✅ Resumed from epoch {start_epoch} | Best mIoU: {best_miou:.4f}")
else:
    print("🆕 Starting fresh")

# ── mIoU ─────────────────────────────────────────────────────────────────────
def compute_miou(model, val_dl, device, num_classes=NUM_CLASSES):
    intersection = np.zeros(num_classes)
    union        = np.zeros(num_classes)
    model.eval()
    with torch.no_grad():
        for imgs, anns in tqdm(val_dl, desc='  Val mIoU', leave=False):
            imgs = imgs.to(device)
            out  = model(pixel_values=imgs)
            upsampled = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE),
                                      mode='bilinear', align_corners=False)
            preds = upsampled.argmax(dim=1).cpu().numpy()
            anns  = anns.squeeze(1).numpy()
            for c in range(num_classes):
                pred_c  = preds == c
                label_c = anns  == c
                intersection[c] += (pred_c & label_c).sum()
                union[c]        += (pred_c | label_c).sum()
    iou_per_class = np.where(union > 0, intersection / union, np.nan)
    return float(np.nanmean(iou_per_class))

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, (imgs, anns) in enumerate(pbar):
        imgs = imgs.to(DEVICE)
        anns = anns.squeeze(1).to(DEVICE)

        with autocast('cuda'):
            out       = model(pixel_values=imgs)
            logits_up = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE),
                                      mode='bilinear', align_corners=False)
            loss      = F.cross_entropy(logits_up, anns)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        # Mid-epoch save every 500 steps — local only, fast
        if (step + 1) % 500 == 0:
            torch.save({
                'epoch':     epoch,
                'model':     model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler':    scaler.state_dict(),
                'best_miou': best_miou,
            }, RESUME_PATH)
            print(f"  💾 Mid-epoch checkpoint saved (step {step+1})")

    scheduler.step()
    miou     = compute_miou(model, val_loader, DEVICE)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val mIoU: {miou:.4f}")

    # Save resume checkpoint locally
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler':    scaler.state_dict(),
        'best_miou': best_miou,
    }, RESUME_PATH)

    # Save best model
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ✅ Best model saved (mIoU: {best_miou:.4f})")
        upload_to_drive(BEST_PATH, 'segformer_lip.pth', DRIVE_CKPT_FOLDER_ID)

    # Always upload resume checkpoint to Drive
    upload_to_drive(RESUME_PATH, 'segformer_resume.pth', DRIVE_CKPT_FOLDER_ID)

print(f"\n🎉 Training complete. Best mIoU: {best_miou:.4f}")

ModuleNotFoundError: No module named 'data'

In [29]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pathlib import Path

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

DRIVE_CKPT_FOLDER_ID = '1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8'
CKPT_DIR = Path('/content/Luxe/backend/checkpoints')

def upload_to_drive(local_path, filename, folder_id):
    file_list = drive.ListFile({
        'q': f"title='{filename}' and '{folder_id}' in parents and trashed=false"
    }).GetList()
    if file_list:
        f = drive.CreateFile({'id': file_list[0]['id']})
    else:
        f = drive.CreateFile({'title': filename, 'parents': [{'id': folder_id}]})
    f.SetContentFile(str(local_path))
    f.Upload()
    print(f"✅ Uploaded {filename}")

for fname in ['segformer_lip.pth', 'segformer_resume.pth']:
    path = CKPT_DIR / fname
    if path.exists():
        upload_to_drive(path, fname, DRIVE_CKPT_FOLDER_ID)
    else:
        print(f"❌ Not found: {fname}")

✅ Uploaded segformer_lip.pth
✅ Uploaded segformer_resume.pth


In [1]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pathlib import Path

# Auth
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

DRIVE_CKPT_FOLDER_ID = '1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8'
CKPT_DIR = Path('/content/Luxe/backend/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Download both checkpoints
for fname in ['segformer_resume.pth', 'segformer_lip.pth']:
    file_list = drive.ListFile({
        'q': f"title='{fname}' and '{DRIVE_CKPT_FOLDER_ID}' in parents and trashed=false"
    }).GetList()
    if file_list:
        f = drive.CreateFile({'id': file_list[0]['id']})
        f.GetContentFile(str(CKPT_DIR / fname))
        print(f"✅ Downloaded {fname}")
    else:
        print(f"❌ Not found on Drive: {fname}")

✅ Downloaded segformer_resume.pth
✅ Downloaded segformer_lip.pth


In [3]:
!git clone https://github.com/nabirakhan/luxe /content/Luxe

fatal: destination path '/content/Luxe' already exists and is not an empty directory.


In [16]:
# Re-download the checkpoints from Drive
from pathlib import Path

CKPT_DIR = Path('/content/Luxe/backend/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for fname in ['segformer_resume.pth', 'segformer_lip.pth']:
    file_list = drive.ListFile({
        'q': f"title='{fname}' and '1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8' in parents and trashed=false"
    }).GetList()
    if file_list:
        f = drive.CreateFile({'id': file_list[0]['id']})
        f.GetContentFile(str(CKPT_DIR / fname))
        print(f"✅ Downloaded {fname}")
    else:
        print(f"❌ Not found: {fname}")

✅ Downloaded segformer_resume.pth
✅ Downloaded segformer_lip.pth


In [6]:
import os
print(os.listdir('/content/Luxe'))
print(os.listdir('/content/Luxe/backend'))

['backend']
['checkpoints']


In [7]:
!git -C /content/Luxe log --oneline -5
!git -C /content/Luxe ls-files

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [8]:
!rm -rf /content/Luxe
!git clone https://github.com/nabirakhan/luxe /content/Luxe

Cloning into '/content/Luxe'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 149 (delta 51), reused 99 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 10.69 MiB | 32.37 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [9]:
import os
print(os.listdir('/content/Luxe/backend'))

['texture.py', 'segformer.py', 'clip_attack.py', 'data', 'main.py', 'eot.py', 'protect.py', '.gitignore', 'metrics.py', 'unet.py', 'train_unet.py', 'utils.py', 'config.py', 'eval_runner.py', 'requirements.txt', 'pgd_modification.py', 'training', 'pgd_inpaint.py']


In [12]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [13]:
import os

os.makedirs('/content/lip/images', exist_ok=True)
os.makedirs('/content/lip/annotations', exist_ok=True)
os.makedirs('/content/deepfashion', exist_ok=True)

files_to_download = {
    'TrainVal_images.zip':              ('11056WM2HM1C2WGzOWoA1NnIwPucbM0Qa', '/content/lip/images/'),
    'TrainVal_parsing_annotations.zip': ('1srpJV9OIT7C-qK-FI4ifETzSVdSGviRk', '/content/lip/annotations/'),
    'train_id.txt':                     ('1-0lHlpNwGqH-CbOBml07eMv5zc1NCMXg', '/content/lip/'),
    'val_id.txt':                       ('1BSCGeJX6KNICWvSj17EzEtMQiCGrP21e',  '/content/lip/'),
}

for name, (file_id, dest) in files_to_download.items():
    out_path = dest + name
    if os.path.exists(out_path):
        print(f"✅ Already exists: {name}")
        continue
    print(f"Downloading {name}...")
    f = drive.CreateFile({'id': file_id})
    f.GetContentFile(out_path)
    size = os.path.getsize(out_path) // (1024*1024)
    print(f"✅ Done — {size}MB → {out_path}")

✅ Done — 628MB → /content/lip/images/TrainVal_images.zip
✅ Done — 87MB → /content/lip/annotations/TrainVal_parsing_annotations.zip
✅ Done — 0MB → /content/lip/train_id.txt
✅ Done — 0MB → /content/lip/val_id.txt


In [14]:
import zipfile

print("Extracting images...")
with zipfile.ZipFile('/content/lip/images/TrainVal_images.zip', 'r') as z:
    z.extractall('/content/lip/images/')
print("✅ Images extracted")

print("Extracting annotations...")
with zipfile.ZipFile('/content/lip/annotations/TrainVal_parsing_annotations.zip', 'r') as z:
    z.extractall('/content/lip/annotations/')
print("✅ Annotations extracted")

Extracting images...
✅ Images extracted
Extracting annotations...
✅ Annotations extracted


In [17]:
import os, sys, torch
import numpy as np
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from transformers import SegformerForSemanticSegmentation
from pathlib import Path
from tqdm.auto import tqdm

sys.path.insert(0, '/content/Luxe/backend')
from data.lip_loader import get_lip_loaders

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ── Config ───────────────────────────────────────────────────────────────────
DEVICE               = torch.device('cuda')
NUM_CLASSES          = 20
EPOCHS               = 20
LR                   = 6e-5
BATCH_SIZE           = 4
IMG_SIZE             = 512
DRIVE_CKPT_FOLDER_ID = '1h90ZEcwKbQMBNzKUW2sICpbqZ_XN1Ep8'
CKPT_DIR             = Path('/content/Luxe/backend/checkpoints')
CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH            = CKPT_DIR / 'segformer_lip.pth'
RESUME_PATH          = CKPT_DIR / 'segformer_resume.pth'

# ── Drive upload — never crashes training ─────────────────────────────────────
def upload_to_drive(local_path, filename, folder_id):
    try:
        # Re-auth every time to avoid token expiry
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        fresh_drive = GoogleDrive(gauth)
        
        file_list = fresh_drive.ListFile({
            'q': f"title='{filename}' and '{folder_id}' in parents and trashed=false"
        }).GetList()
        if file_list:
            f = fresh_drive.CreateFile({'id': file_list[0]['id']})
        else:
            f = fresh_drive.CreateFile({'title': filename, 'parents': [{'id': folder_id}]})
        f.SetContentFile(str(local_path))
        f.Upload()
        print(f"  ☁️  Uploaded {filename} to Drive")
    except Exception as e:
        print(f"  ⚠️  Drive upload failed (training continues): {e}")
        
# ── Data ─────────────────────────────────────────────────────────────────────
train_loader, val_loader = get_lip_loaders(
    '/content/lip/images',
    '/content/lip/annotations',
    batch_size=BATCH_SIZE
)
print(f"✅ Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

# ── Model ────────────────────────────────────────────────────────────────────
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/segformer-b2-finetuned-ade-512-512',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler('cuda')

# ── Resume ───────────────────────────────────────────────────────────────────
start_epoch = 0
best_miou   = 0.0

if RESUME_PATH.exists():
    print(f"🔄 Resuming from {RESUME_PATH}")
    ckpt = torch.load(RESUME_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    best_miou   = ckpt['best_miou']
    print(f"✅ Resumed from epoch {start_epoch} | Best mIoU: {best_miou:.4f}")
else:
    print("🆕 Starting fresh")

# ── mIoU ─────────────────────────────────────────────────────────────────────
def compute_miou(model, val_dl, device, num_classes=NUM_CLASSES):
    intersection = np.zeros(num_classes)
    union        = np.zeros(num_classes)
    model.eval()
    with torch.no_grad():
        for imgs, anns in tqdm(val_dl, desc='  Val mIoU', leave=False):
            imgs = imgs.to(device)
            out  = model(pixel_values=imgs)
            upsampled = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE),
                                      mode='bilinear', align_corners=False)
            preds = upsampled.argmax(dim=1).cpu().numpy()
            anns  = anns.squeeze(1).numpy()
            for c in range(num_classes):
                pred_c  = preds == c
                label_c = anns  == c
                intersection[c] += (pred_c & label_c).sum()
                union[c]        += (pred_c | label_c).sum()
    iou_per_class = np.where(union > 0, intersection / union, np.nan)
    return float(np.nanmean(iou_per_class))

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, (imgs, anns) in enumerate(pbar):
        imgs = imgs.to(DEVICE)
        anns = anns.squeeze(1).to(DEVICE)

        with autocast('cuda'):
            out       = model(pixel_values=imgs)
            logits_up = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE),
                                      mode='bilinear', align_corners=False)
            loss      = F.cross_entropy(logits_up, anns)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        # Mid-epoch save every 500 steps — local only, fast
        if (step + 1) % 500 == 0:
            torch.save({
                'epoch':     epoch,
                'model':     model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'scaler':    scaler.state_dict(),
                'best_miou': best_miou,
            }, RESUME_PATH)
            print(f"  💾 Mid-epoch checkpoint saved (step {step+1})")

    scheduler.step()
    miou     = compute_miou(model, val_loader, DEVICE)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val mIoU: {miou:.4f}")

    # Save resume checkpoint locally
    torch.save({
        'epoch':     epoch,
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler':    scaler.state_dict(),
        'best_miou': best_miou,
    }, RESUME_PATH)

    # Save best model
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ✅ Best model saved (mIoU: {best_miou:.4f})")
        upload_to_drive(BEST_PATH, 'segformer_lip.pth', DRIVE_CKPT_FOLDER_ID)

    # Always upload resume checkpoint to Drive
    upload_to_drive(RESUME_PATH, 'segformer_resume.pth', DRIVE_CKPT_FOLDER_ID)

print(f"\n🎉 Training complete. Best mIoU: {best_miou:.4f}")

✅ GPU: Tesla T4
[LIPDataset] split=train  found=30462/30462
[LIPDataset] split=val  found=10000/10000
✅ Train: 30462 | Val: 10000


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                    
------------------------------+----------+----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([20])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([20, 768, 1, 1])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🔄 Resuming from /content/Luxe/backend/checkpoints/segformer_resume.pth
✅ Resumed from epoch 16 | Best mIoU: 0.5259


Epoch 17/20:   0%|          | 0/7616 [00:00<?, ?it/s]

...

Epoch 19/20 | Loss: 0.1533 | Val mIoU: 0.5283
  ☁️  Uploaded segformer_resume.pth to Drive


Epoch 20/20:   0%|          | 0/7616 [00:00<?, ?it/s]

  💾 Mid-epoch checkpoint saved (step 500)
  💾 Mid-epoch checkpoint saved (step 1000)
  💾 Mid-epoch checkpoint saved (step 1500)
  💾 Mid-epoch checkpoint saved (step 2000)
  💾 Mid-epoch checkpoint saved (step 2500)
  💾 Mid-epoch checkpoint saved (step 3000)
  💾 Mid-epoch checkpoint saved (step 3500)
  💾 Mid-epoch checkpoint saved (step 4000)
  💾 Mid-epoch checkpoint saved (step 4500)
  💾 Mid-epoch checkpoint saved (step 5000)
  💾 Mid-epoch checkpoint saved (step 5500)
  💾 Mid-epoch checkpoint saved (step 6000)
  💾 Mid-epoch checkpoint saved (step 6500)
  💾 Mid-epoch checkpoint saved (step 7000)
  💾 Mid-epoch checkpoint saved (step 7500)


  Val mIoU:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 20/20 | Loss: 0.1467 | Val mIoU: 0.5308
  ✅ Best model saved (mIoU: 0.5308)
  ☁️  Uploaded segformer_lip.pth to Drive
  ☁️  Uploaded segformer_resume.pth to Drive

🎉 Training complete. Best mIoU: 0.5308
